# H&M Recommendations — Part 2a: Rule-Based Candidate Generation

Pure rule-based candidate generator using 4 strategies:

- **s1** — postal code top-1000 products + segment × age fallback (fills remaining to 300)
- **s2a** — recent purchase history, recency-sorted (15 slots)
- **s2b** — colour variants of purchased items, ranked by popularity (15 slots)
- **s3** — frequently bought together / co-occurrence (30 slots)

Slot sizes were selected empirically. Postal code strategy uses last-week product popularity
per region with segment × age as fallback for sparse postal codes.


## 1. Load Data

In [ ]:
from google.colab import drive
drive.mount('/content/drive')


Mounted at /content/drive


In [ ]:
import pandas as pd
import numpy as np
import os
from collections import defaultdict

os.makedirs('/content/drive/MyDrive/HM Fashion/candidates', exist_ok=True)

train     = pd.read_parquet('/content/drive/MyDrive/HM Fashion/processed/train.parquet')
test      = pd.read_parquet('/content/drive/MyDrive/HM Fashion/processed/test.parquet')
customers = pd.read_csv('/content/drive/MyDrive/HM Fashion/processed/customers.csv')
articles  = pd.read_csv('/content/drive/MyDrive/HM Fashion/processed/articles.csv',
                        dtype={'article_id': str})

ground_truth = test.groupby('customer_id')['article_id'].apply(set).reset_index()
ground_truth.columns = ['customer_id', 'purchased']

split_date = pd.to_datetime(train['t_dat'].max())
test_users = test['customer_id'].unique()

print(f'Train     : {len(train):,}')
print(f'Test      : {len(test):,}')
print(f'Test users: {len(test_users):,}')


Train     : 31,548,013
Test      : 240,311
Test users: 68,984


## 2. Filter Recently Active Items

Only consider items bought by anyone in the last 6 weeks. Removes dead catalog items.

In [ ]:
six_weeks_ago = split_date - pd.Timedelta(weeks=6)
one_week_ago  = split_date - pd.Timedelta(weeks=1)

recently_active = set(train[train['t_dat'] >= six_weeks_ago]['article_id'].unique())
last_week_items = set(train[train['t_dat'] >= one_week_ago]['article_id'].unique())
train_recent    = train[train['article_id'].isin(recently_active)].copy()

print(f'Total catalog items  : {train.article_id.nunique():,}')
print(f'Recently active (6w) : {len(recently_active):,}')
print(f'Active last week     : {len(last_week_items):,}')
print(f'Train after filter   : {len(train_recent):,} transactions')


Total catalog items  : 103,880
Recently active (6w) : 33,146
Active last week     : 19,333
Train after filter   : 19,006,198 transactions


## 3. User Features - Age, Segment, Gender

In [ ]:
# age buckets
def age_bucket(age):
    if pd.isna(age):  return 'unknown'
    elif age < 18:    return 'under_18'
    elif age < 26:    return 'age_18_25'
    elif age < 36:    return 'age_26_35'
    elif age < 51:    return 'age_36_50'
    elif age < 66:    return 'age_51_65'
    else:             return 'age_65_plus'

customers['age_group'] = customers['age'].apply(age_bucket)
user_age_group         = customers.set_index('customer_id')['age_group'].to_dict()

last_week_train = train[train['t_dat'] >= one_week_ago].copy()
last_week_train['age_group'] = last_week_train['customer_id'].map(user_age_group).fillna('unknown')

print('Age buckets: under_18, age_18_25, age_26_35, age_36_50, age_51_65, age_65_plus')


Age buckets: under_18, age_18_25, age_26_35, age_36_50, age_51_65, age_65_plus


In [ ]:
# derive user primary segment from purchase history
section_to_segment = {
    'Womens Everyday Collection': 'woman', 'Womens Casual': 'woman',
    'Womens Trend': 'woman', 'Womens Premium': 'woman',
    'Womens Tailoring': 'woman', 'Womens Jackets': 'woman',
    'Womens Everyday Basics': 'woman', 'Ladies Denim': 'woman',
    'Womens Lingerie': 'woman', 'Womens Swimwear, beachwear': 'woman',
    'Womens Nightwear, Socks & Tigh': 'woman', 'Womens Small accessories': 'woman',
    'Womens Big accessories': 'woman', 'Ladies Other': 'woman',
    'H&M+': 'woman_plus_size', 'Mama': 'pregnant_woman',
    'Ladies H&M Sport': 'woman_sport',
    'Contemporary Casual': 'man', 'Contemporary Smart': 'man',
    'Contemporary Street': 'man', 'Men Suits & Tailoring': 'man',
    'Men Underwear': 'man', 'Mens Outerwear': 'man',
    'Men Accessories': 'man', 'Denim Men': 'man',
    'Men Edition': 'man', 'Men Project': 'man',
    'Men Other': 'man', 'Men Other 2': 'man', 'Men Shoes': 'man',
    'Men H&M Sport': 'man_sport',
    'Divided Collection': 'young_woman', 'Divided Basics': 'young_woman',
    'Divided Projects': 'young_woman', 'Divided Accessories': 'young_woman',
    'Divided Selected': 'young_woman', 'Divided Asia keys': 'young_woman',
    'Divided Complements Other': 'young_woman', 'EQ Divided': 'young_woman',
    'Young Girl': 'girl_teenager', 'Young Boy': 'boy_teenager',
    'Kids Girl': 'parent_girl_medium', 'Kids Boy': 'parent_boy_medium',
    'Kids Outerwear': 'parent_kid', 'Kids Accessories, Swimwear & D': 'parent_kid',
    'Kids Local Relevance': 'parent_kid', 'Baby Girl': 'parent_girl_small',
    'Baby Boy': 'parent_boy_small', 'Baby Essentials & Complements': 'parent_baby',
    'Boys Underwear & Basics': 'parent_boy_medium',
    'Girls Underwear & Basics': 'parent_girl_medium',
    'Kids Sports': 'parent_kid_sport',
    'Special Collections': 'fashion_forward', 'Collaborations': 'fashion_forward',
}

train_with_section = train.merge(
    articles[['article_id', 'section_name']], on='article_id', how='left')
train_with_section['segment'] = train_with_section['section_name'].map(section_to_segment)

user_primary_segment = (train_with_section.dropna(subset=['segment'])
                                            .groupby(['customer_id', 'segment'])
                                            .size().reset_index(name='cnt')
                                            .sort_values('cnt', ascending=False)
                                            .groupby('customer_id').first()
                                            .reset_index()
                                            .set_index('customer_id')['segment']
                                            .to_dict())

print(f'Users with segment: {len(user_primary_segment):,}')
print(pd.Series(user_primary_segment).value_counts().head(10).to_string())


Users with segment: 1,349,062
woman                 994179
young_woman           174200
man                    63577
woman_sport            29366
pregnant_woman         17359
woman_plus_size        16895
parent_girl_medium     12565
parent_boy_medium      10333
girl_teenager           7909
boy_teenager            5775


In [ ]:
# derive gender from segment
segment_to_gender = {
    'woman': 'female', 'young_woman': 'female', 'woman_plus_size': 'female',
    'woman_sport': 'female', 'pregnant_woman': 'female', 'girl_teenager': 'female',
    'man': 'male', 'man_sport': 'male', 'boy_teenager': 'male',
    'parent_girl_medium': 'parent', 'parent_boy_medium': 'parent',
    'parent_girl_small': 'parent', 'parent_boy_small': 'parent',
    'parent_kid': 'parent', 'parent_kid_sport': 'parent',
    'parent_baby': 'parent', 'fashion_forward': 'female',
}

user_gender = {u: segment_to_gender.get(user_primary_segment.get(u, 'woman'), 'female')
               for u in train['customer_id'].unique()}

print('User gender distribution:')
print(pd.Series(user_gender).value_counts().to_string())


User gender distribution:
female    1249334
male        70283
parent      37092


## 4. Strategy s1 — Postal Code Top-1000 Products + Segment × Age Fallback

Primary: top-1000 products from last week for user's postal code (mapped to most popular article per product_code).
Fallback: segment × age popularity fills any remaining slots when postal is sparse or missing.

EDA showed postal code top-1000 products cover 99.8% of transactions - likely reflects regional availability.
Segment × age fallback (46.5% coverage) is used when postal code has insufficient data.

In [ ]:
# add gender and age_group to last_week_train
last_week_train['gender']    = last_week_train['customer_id'].map(user_gender).fillna('female')
last_week_train['age_group'] = last_week_train['customer_id'].map(user_age_group).fillna('unknown')

# user postal code
user_postal = customers.set_index('customer_id')['postal_code'].to_dict()
last_week_train['postal_code'] = last_week_train['customer_id'].map(user_postal)

# ── postal top-1000 products → articles ──────────────────────────────────
last_week_with_pcode = last_week_train.merge(
    articles[['article_id', 'product_code']], on='article_id', how='left')

# top-1000 product_codes per postal code from last week
postal_top_pcodes = (last_week_with_pcode
    .groupby(['postal_code', 'product_code']).size()
    .reset_index(name='cnt').sort_values('cnt', ascending=False)
    .groupby('postal_code').head(1000))

# map each product_code to its most popular article in last week
pcode_top_article = (last_week_with_pcode
    .groupby(['product_code', 'article_id']).size()
    .reset_index(name='cnt').sort_values('cnt', ascending=False)
    .groupby('product_code').first()['article_id'].to_dict())

postal_top_articles = {}
for postal, grp in postal_top_pcodes.groupby('postal_code'):
    arts = [pcode_top_article[pc] for pc in grp['product_code']
            if pc in pcode_top_article
            and pcode_top_article[pc] in recently_active]
    if arts:
        postal_top_articles[postal] = arts

# ── segment × age fallback ────────────────────────────────────────────────
last_week_train['segment'] = last_week_train['customer_id'].map(user_primary_segment).fillna('woman')

top_by_segment_age_dict = (last_week_train[last_week_train['article_id'].isin(recently_active)]
    .groupby(['segment', 'age_group', 'article_id'])
    .size().reset_index(name='count')
    .sort_values('count', ascending=False)
    .groupby(['segment', 'age_group']).head(400)
    .groupby(['segment', 'age_group'])['article_id'].apply(list).to_dict())

# global gender fallback for unknown segments
global_top_female = (last_week_train[last_week_train['gender'] == 'female']
    [last_week_train['article_id'].isin(recently_active)]
    .groupby('article_id').size()
    .sort_values(ascending=False).head(400).index.tolist())
global_top_male   = (last_week_train[last_week_train['gender'] == 'male']
    [last_week_train['article_id'].isin(recently_active)]
    .groupby('article_id').size()
    .sort_values(ascending=False).head(400).index.tolist())
global_top_parent = (last_week_train[last_week_train['gender'] == 'parent']
    [last_week_train['article_id'].isin(recently_active)]
    .groupby('article_id').size()
    .sort_values(ascending=False).head(400).index.tolist())
global_top        = (last_week_train[last_week_train['article_id'].isin(recently_active)]
    .groupby('article_id').size()
    .sort_values(ascending=False).head(400).index.tolist())

print(f'Postal codes with top products  : {len(postal_top_articles):,}')
print(f'Segment × age groups            : {len(top_by_segment_age_dict):,}')
postal_coverage = sum(1 for u in last_week_train["customer_id"].unique()
                      if user_postal.get(u) in postal_top_articles)
print(f'Users with postal coverage      : {postal_coverage:,}')


/tmp/ipykernel_2927/2574366928.py:44: UserWarning: Boolean Series key will be reindexed to match DataFrame index.
  global_top_female = (last_week_train[last_week_train['gender'] == 'female']
/tmp/ipykernel_2927/2574366928.py:48: UserWarning: Boolean Series key will be reindexed to match DataFrame index.
  global_top_male   = (last_week_train[last_week_train['gender'] == 'male']
/tmp/ipykernel_2927/2574366928.py:52: UserWarning: Boolean Series key will be reindexed to match DataFrame index.
  global_top_parent = (last_week_train[last_week_train['gender'] == 'parent']


Postal codes with top products  : 66,062
Segment × age groups            : 86
Users with postal coverage      : 80,765


## 5. Strategies s2a & s2b — Purchase History + Colour Variants

s1a: Purchase history sorted by recency - items user bought recently (15 slots).
s1b: Colour variant repurchase - same product_code, different colour, cap=1, ranked by global popularity (15 slots).

In [ ]:
# purchase history - deduplicated, sorted by recency
# used for s1a (exact), s2 (bought-together) and s1b (variants)
user_history = (train_recent
                .sort_values('t_dat', ascending=False)
                .groupby('customer_id')['article_id']
                .apply(lambda x: list(dict.fromkeys(x)))
                .to_dict())

# article popularity for variant ranking
article_popularity = train.groupby('article_id').size().to_dict()

# build lookups
article_to_pcode        = articles.set_index('article_id')['product_code'].to_dict()
article_to_colour_group = articles.set_index('article_id')['colour_group_code'].to_dict()
pcode_all_articles      = articles.groupby('product_code')['article_id'].apply(list).to_dict()
article_to_type         = articles.set_index('article_id')['product_type_name'].to_dict()

print(f'Users with purchase history : {len(user_history):,}')
print(f'Avg history items per user  : {sum(len(v) for v in user_history.values()) / len(user_history):.1f}')
print(f'Product codes               : {len(pcode_all_articles):,}')


Users with purchase history : 1,206,461
Avg history items per user  : 13.3
Product codes               : 47,224


## 6. Strategy s3 — Frequently Bought Together (Co-occurrence)

Items bought by the same user within the same week. Co-occurrence matrix built on recent transactions.

In [ ]:
train_recent['year_week'] = (train_recent['t_dat'].dt.year.astype(str) + '_' +
                             train_recent['t_dat'].dt.isocalendar().week.astype(str))

# deduplicate - keep unique items per user per week
# prevents same item counting as co-occurrence with itself
user_week_items = (train_recent
                   .drop_duplicates(subset=['customer_id', 'year_week', 'article_id'])
                   .groupby(['customer_id', 'year_week'])['article_id']
                   .apply(list).reset_index())

print('Building co-occurrence matrix...')
co_occurrence = defaultdict(lambda: defaultdict(int))

for _, row in user_week_items.iterrows():
    items = row['article_id']
    if len(items) < 2:
        continue
    for i in range(len(items)):
        for j in range(i+1, len(items)):
            co_occurrence[items[i]][items[j]] += 1
            co_occurrence[items[j]][items[i]] += 1

bought_together = {}
for item, co_items in co_occurrence.items():
    top_co = sorted(co_items.items(), key=lambda x: x[1], reverse=True)[:250]
    bought_together[item] = [a for a, _ in top_co
                              if a in recently_active
                              and a != item]  # exclude self

# save co-occurrence for reranker feature
co_occurrence_rows = []
for item_a, co_items in co_occurrence.items():
    for item_b, count in co_items.items():
        if item_a in recently_active and item_b in recently_active:
            co_occurrence_rows.append({'article_id_a': item_a,
                                       'article_id_b': item_b,
                                       'co_count':     count})

co_occurrence_df = pd.DataFrame(co_occurrence_rows)
co_occurrence_df.to_parquet(
    '/content/drive/MyDrive/HM Fashion/processed/co_occurrence.parquet', index=False)

print(f'Items with co-occurrence data : {len(bought_together):,}')
print(f'Avg co-occurring items        : {np.mean([len(v) for v in bought_together.values()]):.1f}')
print(f'Co-occurrence pairs saved     : {len(co_occurrence_df):,}')

Building co-occurrence matrix...
Items with co-occurrence data : 33,081
Avg co-occurring items        : 200.0
Co-occurrence pairs saved     : 36,149,216


## 7. Generate Candidates

In [ ]:
def generate_rule_candidates(test_users, total_k=300):
    results = []
    for user_id in test_users:
        history   = user_history.get(user_id, [])
        gender    = user_gender.get(user_id, 'female')
        age_group = user_age_group.get(user_id, 'unknown')
        segment   = user_primary_segment.get(user_id, 'woman')
        postal    = user_postal.get(user_id)

        # ── s1a: purchase history sorted by recency (15 slots) ────────────
        s1_exact = list(history[:15])

        # ── s1b: colour variants cap=1 ranked by popularity (15 slots) ────
        s1_variants = []
        seen_pcodes = set()
        for item in history:
            if len(s1_variants) >= 15:
                break
            pcode  = article_to_pcode.get(item)
            colour = article_to_colour_group.get(item)
            if pcode and colour and pcode not in seen_pcodes:
                all_variants = pcode_all_articles.get(pcode, [])
                diff_colour  = sorted(
                    [v for v in all_variants
                     if v != item
                     and article_to_colour_group.get(v) != colour
                     and v in recently_active],
                    key=lambda v: article_popularity.get(v, 0),
                    reverse=True)[:1]
                s1_variants.extend(diff_colour)
                seen_pcodes.add(pcode)

        # ── s2: bought together (30 slots) ────────────────────────────────
        bt_items = []
        for item in history[:10]:
            bt_items.extend(bought_together.get(item, []))
        s2 = list(dict.fromkeys(bt_items))[:30]

        # ── combine s1a + s1b + s2, deduplicate ───────────────────────────
        seen       = set()
        candidates = []
        for c in s1_exact + s1_variants + s2:
            if c not in seen and c in recently_active:
                seen.add(c)
                candidates.append(c)

        # ── s4: postal top-1000 primary + segment×age fills remaining ─────
        postal_items   = postal_top_articles.get(postal, [])
        seg_age_items  = top_by_segment_age_dict.get(
            (segment, age_group),
            global_top_female if gender == 'female'
            else global_top_male if gender == 'male'
            else global_top_parent)

        for c in postal_items + seg_age_items:
            if c not in seen and c in recently_active:
                seen.add(c)
                candidates.append(c)
            if len(candidates) >= total_k:
                break

        # ── final global fallback if still not enough ─────────────────────
        if len(candidates) < total_k:
            for c in global_top:
                if c not in seen and c in recently_active:
                    seen.add(c)
                    candidates.append(c)
                if len(candidates) >= total_k:
                    break

        if candidates:
            results.append({'customer_id': user_id,
                            'article_id':  candidates[:total_k]})

    return pd.DataFrame(results)

test_users      = test['customer_id'].unique()
rule_candidates = generate_rule_candidates(test_users, total_k=300)

print(f'Generated candidates for {len(rule_candidates):,} users')
print(f'Avg candidates per user : {rule_candidates["article_id"].apply(len).mean():.1f}')
postal_used = sum(1 for u in test_users if user_postal.get(u) in postal_top_articles)
print(f'Users with postal s4    : {postal_used:,} ({postal_used/len(test_users):.1%})')
print(f'Users using seg×age only: {len(test_users)-postal_used:,} ({(len(test_users)-postal_used)/len(test_users):.1%})')


Generated candidates for 68,984 users
Avg candidates per user : 300.0
Users with postal s4    : 30,221 (43.8%)
Users using seg×age only: 38,763 (56.2%)


## 8. Evaluation — Recall@300

In [ ]:
merged = rule_candidates.merge(ground_truth, on='customer_id', how='inner')
merged['hits']   = merged.apply(
    lambda row: len(set(row['article_id']) & row['purchased']), axis=1)
merged['recall'] = merged.apply(
    lambda row: len(set(row['article_id']) & row['purchased']) / len(row['purchased']),
    axis=1)

avg_purchased = test.groupby('customer_id')['article_id'].count().mean()

print(f'Recall (all candidates)          : {merged["recall"].mean():.4f}')
print(f'Avg items purchased in test week : {avg_purchased:.2f}')
print(f'Avg relevant items in candidates : {merged["hits"].mean():.3f}')
print(f'Users with 0 relevant items      : {(merged["hits"] == 0).mean():.1%}')
print(f'Users evaluated                  : {len(merged):,} out of {len(test_users):,}')
print()
print('Distribution of hits:')
print(merged['hits'].value_counts().sort_index().head(8).to_string())


Recall (all candidates)          : 0.2762
Avg items purchased in test week : 3.48
Avg relevant items in candidates : 0.796
Users with 0 relevant items      : 51.7%
Users evaluated                  : 68,984 out of 68,984

Distribution of hits:
hits
0    35657
1    20835
2     7536
3     2795
4     1156
5      528
6      243
7      115


## 9. Per-Strategy Contribution

In [ ]:
def eval_single_strategy(candidates_dict, ground_truth, test_users):
    results = []
    for user_id in test_users:
        items = candidates_dict.get(user_id, [])
        results.append({'customer_id': user_id, 'article_id': items})
    df     = pd.DataFrame(results)
    merged = df.merge(ground_truth, on='customer_id', how='inner')
    merged['hits'] = merged.apply(
        lambda row: len(set(row['article_id']) & row['purchased']), axis=1)
    recall = merged.apply(
        lambda row: len(set(row['article_id']) & row['purchased']) / len(row['purchased']),
        axis=1).mean()
    return recall, merged['hits'].mean(), (merged['hits'] == 0).mean()

# ── s1a ───────────────────────────────────────────────────────────────────
rep_dict = {u: user_history.get(u, [])[:15] for u in test_users}
r1, h1, z1 = eval_single_strategy(rep_dict, ground_truth, test_users)

# ── s1b ───────────────────────────────────────────────────────────────────
var_dict = {}
for u in test_users:
    items       = []
    seen_pcodes = set()
    for item in user_history.get(u, []):
        if len(items) >= 15: break
        pcode  = article_to_pcode.get(item)
        colour = article_to_colour_group.get(item)
        if pcode and colour and pcode not in seen_pcodes:
            all_variants = pcode_all_articles.get(pcode, [])
            diff_colour  = sorted(
                [v for v in all_variants
                 if v != item
                 and article_to_colour_group.get(v) != colour
                 and v in recently_active],
                key=lambda v: article_popularity.get(v, 0),
                reverse=True)[:1]
            items.extend(diff_colour)
            seen_pcodes.add(pcode)
    var_dict[u] = items
r1b, h1b, z1b = eval_single_strategy(var_dict, ground_truth, test_users)

# ── s2 ────────────────────────────────────────────────────────────────────
bt_dict = {}
for u in test_users:
    items = []
    for item in user_history.get(u, [])[:10]:
        items.extend(bought_together.get(item, []))
    bt_dict[u] = list(dict.fromkeys(items))[:30]
r2, h2, z2 = eval_single_strategy(bt_dict, ground_truth, test_users)

# ── s4: postal primary + segment×age fallback ─────────────────────────────
pop_dict = {}
for u in test_users:
    postal    = user_postal.get(u)
    segment   = user_primary_segment.get(u, 'woman')
    age_group = user_age_group.get(u, 'unknown')
    gender    = user_gender.get(u, 'female')
    postal_items  = postal_top_articles.get(postal, [])
    seg_age_items = top_by_segment_age_dict.get(
        (segment, age_group),
        global_top_female if gender == 'female'
        else global_top_male if gender == 'male'
        else global_top_parent)
    pop_dict[u] = (postal_items + seg_age_items)[:1000]
r4, h4, z4 = eval_single_strategy(pop_dict, ground_truth, test_users)

print(f'{"Strategy":<52} {"Recall":>8} {"Avg hits":>10} {"Zero-hit%":>10}')
print('-' * 84)
print(f'{"1a. Purchase history recency (15 slots)":<52} {r1:>8.4f} {h1:>10.3f} {z1:>9.1%}')
print(f'{"1b. Colour variants (15 slots)":<52} {r1b:>8.4f} {h1b:>10.3f} {z1b:>9.1%}')
print(f'{"2.  Bought together (30 slots)":<52} {r2:>8.4f} {h2:>10.3f} {z2:>9.1%}')
print(f'{"4.  Postal top-1000 + seg×age fallback":<52} {r4:>8.4f} {h4:>10.3f} {z4:>9.1%}')
print()
print(f'{"Combined (all strategies)":<52} {merged["recall"].mean():>8.4f} {merged["hits"].mean():>10.3f} {(merged["hits"]==0).mean():>9.1%}')


Strategy                                               Recall   Avg hits  Zero-hit%
------------------------------------------------------------------------------------
1a. Purchase history recency (15 slots)                0.0426      0.096     92.2%
1b. Colour variants (15 slots)                         0.0147      0.039     96.3%
2.  Bought together (30 slots)                         0.0321      0.083     93.1%
4.  Postal top-1000 + seg×age fallback                 0.2993      0.892     49.0%

Combined (all strategies)                              0.2762      0.796     51.7%


## 10. Cold-Start Analysis

In [ ]:
purchase_counts = train.groupby('customer_id').size().rename('purchase_count')

all_test_users = ground_truth[['customer_id']].copy()
merged_full    = all_test_users.merge(rule_candidates, on='customer_id', how='left')
merged_full    = merged_full.merge(ground_truth, on='customer_id', how='left')
merged_full    = merged_full.merge(purchase_counts.reset_index(), on='customer_id', how='left')
merged_full['is_cold']    = merged_full['purchase_count'].fillna(0) <= 5
merged_full['article_id'] = merged_full['article_id'].apply(lambda x: x if isinstance(x, list) else [])
merged_full['hits']       = merged_full.apply(
    lambda row: len(set(row['article_id']) & row['purchased']), axis=1)

print(f'Avg hits - warm users : {merged_full[~merged_full["is_cold"]]["hits"].mean():.3f}')
print(f'Avg hits - cold users : {merged_full[merged_full["is_cold"]]["hits"].mean():.3f}')
print(f'Zero hits - warm      : {(merged_full[~merged_full["is_cold"]]["hits"] == 0).mean():.1%}')
print(f'Zero hits - cold      : {(merged_full[merged_full["is_cold"]]["hits"] == 0).mean():.1%}')


Avg hits - warm users : 0.824
Avg hits - cold users : 0.644
Zero hits - warm      : 50.7%
Zero hits - cold      : 56.8%


## 11. Save Candidates

In [ ]:
rule_candidates.to_parquet(
    '/content/drive/MyDrive/HM Fashion/candidates/rule_candidates.parquet', index=False)

print(f'Saved rule_candidates.parquet  ({len(rule_candidates):,} users)')


Saved rule_candidates.parquet  (68,984 users)
